In [1]:
from __future__ import annotations
import matplotlib.pyplot as plt

from pathlib import Path

import json

import numpy as np

import pandas as pd

import geopandas as gpd

from shapely.geometry import box, mapping

# ---- Satellite querying (Planetary Computer STAC) ----
# Make this cell robust to missing packages inside the *notebook kernel*.
try:
    from pystac_client import Client
    import planetary_computer as pc
except ModuleNotFoundError:
    import sys
    import subprocess
    print('Installing missing satellite deps into kernel...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pystac-client', 'planetary-computer'])
    from pystac_client import Client
    import planetary_computer as pc

import rasterio
import rasterio.mask

import matplotlib
matplotlib.use('Agg')  # never inline images


def _find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for p in [start] + list(start.parents):
        # Heuristic: repo root should contain output/ and src/
        if (p / 'output').exists() and (p / 'src').exists():
            return p
        if (p / 'Readme.md').exists() or (p / 'README.md').exists():
            return p
    return start


REPO_ROOT = _find_repo_root(Path.cwd())
OUTPUT_DIR = REPO_ROOT / 'output'
print('REPO_ROOT:', REPO_ROOT)
print('OUTPUT_DIR:', OUTPUT_DIR)

# ---- User-configurable parameters ----
DATASET_NAME = 'sit'  # label for outputs

# Experiment A: cloud-filter sweep

# If None, query items WITHOUT cloud filtering and filter in pandas later (recommended for sweeps).
MAX_CLOUD_PCT_QUERY: int | None = None

# The cloud threshold actually used downstream (best-item selection + NDVI TS).
MAX_CLOUD_PCT_USE: int | None = 60

# Cloud-thresholds to compare for data availability diagnostics (set [] to skip).
CLOUD_THRESHOLDS_EXPERIMENT: list[int | None] = [20, 40, 60, 80, None]

# Whether to keep only 1 item per date (lowest cloud_cover).
ONE_ITEM_PER_DATE = True

# Experiment B: multi-year harmonization
USE_MULTI_YEAR = True
YEARS = [2022, 2023, 2024]  # start with last 3 available years
YEAR = 2024  # used if USE_MULTI_YEAR=False

# Query date bounds for each year (full year by default).
START_MM_DD = '01-01'
END_MM_DD = '12-31'

# Weekly harmonization controls
WEEKLY_MIN_OBS_PER_CROWN_WEEK = 1

# If None, we will auto-pick a likely crowns file from output/
CROWNS_PATH: str | None = None

# Where to write plots/CSVs produced by this notebook
run_label = f"{DATASET_NAME}_{'multi' if USE_MULTI_YEAR else YEAR}_run"
RUN_DIR = OUTPUT_DIR / 'sat_data' / run_label
RUN_DIR.mkdir(parents=True, exist_ok=True)
print('RUN_DIR:', RUN_DIR.resolve())

REPO_ROOT: /Users/hbot07/VS Code/Drone-Phenology-Monitoring
OUTPUT_DIR: /Users/hbot07/VS Code/Drone-Phenology-Monitoring/output
RUN_DIR: /Users/hbot07/VS Code/Drone-Phenology-Monitoring/output/sat_data/sit_multi_run


In [2]:
def _pick_default_crowns_path(output_dir: Path) -> Path:

    # If we're working with the SIT dataset, prefer the SIT tracking run's crowns.

    # This matches the crown set used by the interactive SIT viewer (in georeferenced coords).

    if DATASET_NAME.lower() == 'sit':

        sit_geojson = output_dir / 'sit_tracking_rerun_1Apr26' / \
            'consensus_crowns_om1_phenology_sit.geojson'

        if sit_geojson.exists():

            return sit_geojson

    # Preference order: tree-only / phenology-ready / consensus crowns.

    candidates = [

        output_dir / 'consensus_crowns_tree_only.gpkg',

        output_dir / 'consensus_crowns_complete_all_tree_only.gpkg',

        output_dir / 'consensus_crowns_complete_all.gpkg',

        output_dir / 'consensus_crowns_tree_only_hard_rule.gpkg',

        output_dir / 'consensus_crowns_om1_phenology.geojson',

        output_dir / 'consensus_crowns_all_chains.gpkg',

    ]

    for p in candidates:

        if p.exists():

            return p

    # Last resort: any file matching consensus crowns

    for p in sorted(output_dir.glob('consensus_crowns*.*')):

        if p.suffix.lower() in {'.gpkg', '.geojson', '.json'}:

            return p

    raise FileNotFoundError(

        f'Could not auto-find a crowns file under {output_dir}. Set CROWNS_PATH explicitly.'

    )


crowns_path = Path(
    CROWNS_PATH) if CROWNS_PATH else _pick_default_crowns_path(OUTPUT_DIR)

print('Using crowns file:', crowns_path)


# If GPKG, we may need to choose a layer; try common ones and fall back to first.

crowns: gpd.GeoDataFrame

if crowns_path.suffix.lower() == '.gpkg':

    layers = list(gpd.list_layers(crowns_path).name)

    preferred = [

        'crowns',

        'consensus_crowns',

        'tree_only',

        'crowns_tree_only',

    ]

    layer = next((ly for ly in preferred if ly in layers),
                 layers[0] if layers else None)

    if layer is None:

        raise ValueError(f'No layers found in {crowns_path}')

    print('Using layer:', layer)

    crowns = gpd.read_file(crowns_path, layer=layer)

else:

    crowns = gpd.read_file(crowns_path)


crowns = crowns[crowns.geometry.notnull() & ~crowns.geometry.is_empty].copy()

crowns = crowns.reset_index(drop=True)

print('n_crowns:', len(crowns))

print('crowns.crs:', crowns.crs)


bounds = crowns.total_bounds  # [minx, miny, maxx, maxy]

print('crowns.bounds:', bounds)


# Build an AOI bbox and also a WGS84 bbox for STAC search

aoi_poly = box(*bounds)

aoi = gpd.GeoDataFrame({'name': ['aoi']}, geometry=[aoi_poly], crs=crowns.crs)

aoi_wgs84 = aoi.to_crs('EPSG:4326')

minx, miny, maxx, maxy = aoi_wgs84.total_bounds

bbox_wgs84 = [float(minx), float(miny), float(maxx), float(maxy)]


centroid_wgs84 = aoi_wgs84.geometry.iloc[0].centroid

print('AOI centroid (lat, lon):', float(
    centroid_wgs84.y), float(centroid_wgs84.x))

print('AOI bbox WGS84:', bbox_wgs84)


# Persist a small metadata file for reproducibility

if USE_MULTI_YEAR:
    start_date_meta = f"{min(YEARS)}-{START_MM_DD}"
    end_date_meta = f"{max(YEARS)}-{END_MM_DD}"
else:
    start_date_meta = f"{YEAR}-{START_MM_DD}"
    end_date_meta = f"{YEAR}-{END_MM_DD}"

meta = {

    'dataset': DATASET_NAME,

    'year': YEAR,

    'years': YEARS if USE_MULTI_YEAR else [YEAR],

    'start_date': start_date_meta,

    'end_date': end_date_meta,

    'crowns_path': str(crowns_path),

    'crowns_crs': str(crowns.crs),

    'aoi_bbox_wgs84': bbox_wgs84,

    'n_crowns': int(len(crowns)),

}

(RUN_DIR / 'run_meta.json').write_text(json.dumps(meta, indent=2), encoding='utf-8')

print('Wrote:', (RUN_DIR / 'run_meta.json'))

Using crowns file: /Users/hbot07/VS Code/Drone-Phenology-Monitoring/output/sit_tracking_rerun_1Apr26/consensus_crowns_om1_phenology_sit.geojson
n_crowns: 131
crowns.crs: EPSG:32643
crowns.bounds: [ 714282.90172129 3159477.74365683  714516.65620905 3159694.75926381]
AOI centroid (lat, lon): 28.54544367396415 77.19141903694393
AOI bbox WGS84: [77.19020487434051, 28.544445659547318, 77.19263322195405, 28.54644168390476]
Wrote: /Users/hbot07/VS Code/Drone-Phenology-Monitoring/output/sat_data/sit_multi_run/run_meta.json


In [3]:
# ---- 1) Query Sentinel-2 L2A items over our AOI ----

import re

STAC_API = 'https://planetarycomputer.microsoft.com/api/stac/v1'
COLLECTION = 'sentinel-2-l2a'

catalog = Client.open(STAC_API)


def _item_dt_str(it):
    dt = it.datetime
    return dt.strftime('%Y-%m-%d') if dt else None


def _sentinel_epsg_from_item(it, fallback_lat: float) -> int:
    # Prefer explicit proj:epsg if present
    epsg = it.properties.get('proj:epsg')
    if epsg is not None:
        return int(epsg)

    # Derive from MGRS tile / item id. Example tile: '43RGM' or id contains '_T43RGM_'.
    tile = it.properties.get('mgrs:tile')
    zone = None
    if isinstance(tile, str) and len(tile) >= 2 and tile[:2].isdigit():
        zone = int(tile[:2])
    if zone is None:
        m = re.search(r'_T(\d{2})[A-Z]{3}_', it.id)
        if m:
            zone = int(m.group(1))
    if zone is None:
        raise RuntimeError('Could not derive UTM zone for Sentinel item: ' + it.id)

    hemi_north = fallback_lat >= 0
    return (32600 + zone) if hemi_north else (32700 + zone)


def _query_year(year: int) -> list:
    start_date = f'{year}-{START_MM_DD}'
    end_date = f'{year}-{END_MM_DD}'
    query = {}
    if MAX_CLOUD_PCT_QUERY is not None:
        query = {'eo:cloud_cover': {'lt': int(MAX_CLOUD_PCT_QUERY)}}

    search = catalog.search(
        collections=[COLLECTION],
        bbox=bbox_wgs84,
        datetime=f'{start_date}/{end_date}',
        query=query if query else None,
    )
    return list(search.get_items())


def _apply_post_filters(df: pd.DataFrame, max_cloud: int | None, one_item_per_date: bool) -> pd.DataFrame:
    out = df.copy()
    out['cloud_cover'] = pd.to_numeric(out['cloud_cover'], errors='coerce')
    if max_cloud is not None:
        out = out[out['cloud_cover'].notna() & (out['cloud_cover'] <= float(max_cloud))].copy()
    if one_item_per_date:
        out = (out.sort_values(['date', 'cloud_cover', 'id'])
               .groupby('date', as_index=False)
               .first())
    return out.sort_values(['date', 'cloud_cover']).reset_index(drop=True)


years = YEARS if USE_MULTI_YEAR else [YEAR]
items_by_year = {int(y): _query_year(int(y)) for y in years}
items = [it for y in years for it in items_by_year[int(y)]]
print('years:', years)
print('items_found_total:', len(items))
print('items_found_by_year:', {int(y): len(items_by_year[int(y)]) for y in years})

rows = []
fallback_lat = float(centroid_wgs84.y)
for it in items:
    dt = _item_dt_str(it)
    rows.append({
        'id': it.id,
        'datetime': dt,
        'cloud_cover': it.properties.get('eo:cloud_cover'),
        'mgrs:tile': it.properties.get('mgrs:tile'),
        'sat:orbit_state': it.properties.get('sat:orbit_state'),
        'derived_epsg': _sentinel_epsg_from_item(it, fallback_lat=fallback_lat),
    })

items_df_raw = pd.DataFrame(rows)
items_df_raw = items_df_raw.dropna(subset=['datetime']).copy()
items_df_raw['date'] = pd.to_datetime(items_df_raw['datetime'], errors='coerce')
items_df_raw = items_df_raw.dropna(subset=['date']).copy()
items_df_raw['year'] = items_df_raw['date'].dt.year.astype(int)
iso = items_df_raw['date'].dt.isocalendar()
items_df_raw['iso_year'] = iso.year.astype(int)
items_df_raw['iso_week'] = iso.week.astype(int)

items_df = _apply_post_filters(
    items_df_raw,
    max_cloud=MAX_CLOUD_PCT_USE,
    one_item_per_date=ONE_ITEM_PER_DATE,
)

items_raw_csv = RUN_DIR / 'sentinel2_items_raw.csv'
items_df_raw.to_csv(items_raw_csv, index=False)
print('Wrote:', items_raw_csv)

items_csv = RUN_DIR / 'sentinel2_items_filtered.csv'
items_df.to_csv(items_csv, index=False)
print('Wrote:', items_csv)

# Cloud-threshold availability diagnostics
sweep_rows = []
for thr in CLOUD_THRESHOLDS_EXPERIMENT:
    d = _apply_post_filters(
        items_df_raw,
        max_cloud=thr,
        one_item_per_date=ONE_ITEM_PER_DATE,
    )
    if d.empty:
        sweep_rows.append({
            'cloud_threshold': thr if thr is not None else 'none',
            'n_items': 0,
            'n_unique_dates': 0,
            'n_iso_weeks': 0,
            'n_years': 0,
        })
    else:
        sweep_rows.append({
            'cloud_threshold': thr if thr is not None else 'none',
            'n_items': int(len(d)),
            'n_unique_dates': int(d['date'].nunique()),
            'n_iso_weeks': int(d[['iso_year', 'iso_week']].drop_duplicates().shape[0]),
            'n_years': int(d['year'].nunique()),
        })

sweep_df = pd.DataFrame(sweep_rows)
sweep_csv = RUN_DIR / 'cloud_threshold_coverage_summary.csv'
sweep_df.to_csv(sweep_csv, index=False)
print('Wrote:', sweep_csv)
print(sweep_df)

if items_df.empty:
    raise RuntimeError(
        'No Sentinel-2 items left after filtering. Try increasing MAX_CLOUD_PCT_USE or setting MAX_CLOUD_PCT_USE=None.'
    )

# Pick the clearest item for visualization
best_item_id = items_df.iloc[0]['id']
id_to_item = {it.id: it for it in items}
best_item = id_to_item[best_item_id]
best_epsg = int(items_df.iloc[0]['derived_epsg'])
print('best_item:', best_item.id, 'date:', items_df.iloc[0]['date'].strftime('%Y-%m-%d'), 'cloud:', items_df.iloc[0].get('cloud_cover'))
print('best_item derived EPSG:', best_epsg)

/Users/hbot07/anaconda3/envs/detectree/lib/python3.10/site-packages/pystac_client/item_search.py:925: FutureWarning: get_items() is deprecated, use items() instead
  warnings.warn(


years: [2022, 2023, 2024]
items_found_total: 294
items_found_by_year: {2022: 101, 2023: 120, 2024: 73}
Wrote: /Users/hbot07/VS Code/Drone-Phenology-Monitoring/output/sat_data/sit_multi_run/sentinel2_items_raw.csv
Wrote: /Users/hbot07/VS Code/Drone-Phenology-Monitoring/output/sat_data/sit_multi_run/sentinel2_items_filtered.csv
Wrote: /Users/hbot07/VS Code/Drone-Phenology-Monitoring/output/sat_data/sit_multi_run/cloud_threshold_coverage_summary.csv
  cloud_threshold  n_items  n_unique_dates  n_iso_weeks  n_years
0              20      107             107           87        3
1              40      127             127          104        3
2              60      146             146          117        3
3              80      161             161          128        3
4            none      212             212          154        3
best_item: S2B_MSIL2A_20220129T053049_R105_T43RGM_20220213T194758 date: 2022-01-29 cloud: 0.000129
best_item derived EPSG: 32643


In [5]:
# ---- 2) Visualization: RGB underlay + crown outlines (saved to disk) ----


from pyproj import Transformer


def _signed_href(item, asset_key: str) -> str:

    asset = item.assets.get(asset_key)

    if asset is None:

        raise KeyError(f'Missing asset {asset_key!r} on item {item.id}')

    return pc.sign(asset.href)


def _read_window_resampled(src: rasterio.io.DatasetReader, window, max_dim: int = 1400):

    w = int(window.width)

    h = int(window.height)

    if w <= 0 or h <= 0:

        raise ValueError('Empty window')

    scale = max(w / max_dim, h / max_dim, 1.0)

    out_w = max(1, int(round(w / scale)))

    out_h = max(1, int(round(h / scale)))

    data = src.read(

        1,

        window=window,

        out_shape=(out_h, out_w),

        resampling=rasterio.enums.Resampling.bilinear,

    )

    transform = rasterio.windows.transform(window, src.transform)

    # Adjust transform for resampling

    transform = transform * rasterio.Affine.scale(w / out_w, h / out_h)

    return data, transform


def _percentile_stretch(x: np.ndarray, lo=2, hi=98):

    x = x.astype('float32')

    finite = np.isfinite(x)

    if not finite.any():

        return np.zeros_like(x, dtype='float32')

    vmin, vmax = np.percentile(x[finite], [lo, hi])

    if vmax <= vmin:

        return np.clip(x, 0, 1)

    y = (x - vmin) / (vmax - vmin)

    return np.clip(y, 0, 1)


# Sentinel-2 10m True Color: B04 (red), B03 (green), B02 (blue)
href_r = _signed_href(best_item, 'B04')

href_g = _signed_href(best_item, 'B03')

href_b = _signed_href(best_item, 'B02')


# IMPORTANT: rasterio CRS reprojection utilities are broken in this env due to a PROJ db mismatch.

# Use STAC-derived EPSG + pyproj for all coordinate transforms.

s2_epsg = _sentinel_epsg_from_item(
    best_item, fallback_lat=float(centroid_wgs84.y))

s2_crs_str = f'EPSG:{int(s2_epsg)}'


# Transform AOI bbox from WGS84 -> Sentinel CRS (corners are sufficient for small AOI)

minlon, minlat, maxlon, maxlat = bbox_wgs84

tx = Transformer.from_crs('EPSG:4326', s2_crs_str, always_xy=True)

xs, ys = tx.transform([minlon, minlon, maxlon, maxlon],
                      [minlat, maxlat, minlat, maxlat])

aoi_bounds_s2 = (float(min(xs)), float(min(ys)),
                 float(max(xs)), float(max(ys)))


with rasterio.open(href_r) as src_r:

    win = rasterio.windows.from_bounds(
        *aoi_bounds_s2, transform=src_r.transform)

    win = win.round_offsets().round_lengths()

    pad = 40

    win = rasterio.windows.Window(

        col_off=max(0, win.col_off - pad),

        row_off=max(0, win.row_off - pad),

        width=min(src_r.width - max(0, win.col_off - pad),
                  win.width + 2 * pad),

        height=min(src_r.height - max(0, win.row_off - pad),
                   win.height + 2 * pad),

    )


with rasterio.open(href_r) as src_r, rasterio.open(href_g) as src_g, rasterio.open(href_b) as src_b:

    r, t = _read_window_resampled(src_r, win)

    g, _ = _read_window_resampled(src_g, win)

    b, _ = _read_window_resampled(src_b, win)


rgb = np.dstack([

    _percentile_stretch(r),

    _percentile_stretch(g),

    _percentile_stretch(b),

])


# Reproject crowns to Sentinel CRS for plotting (uses pyproj)

crowns_s2 = crowns.to_crs(s2_crs_str)


fig, ax = plt.subplots(figsize=(10, 10), dpi=160)

ax.imshow(rgb)

ax.set_title(
    f'Sentinel-2 RGB + crowns | {DATASET_NAME} | {YEAR} | {s2_crs_str} | item {best_item.id}')

ax.axis('off')


# Convert map coords -> pixel coords in the plotted window

inv_t = ~t


for geom in crowns_s2.geometry:

    if geom is None or geom.is_empty:

        continue

    try:

        geoms = [geom] if geom.geom_type == 'Polygon' else list(
            getattr(geom, 'geoms', []))

        if not geoms:

            continue

        poly = max(geoms, key=lambda p: p.area)

        xs, ys = poly.exterior.xy

        cols_rows = [inv_t * (x, y) for x, y in zip(xs, ys)]

        cols = [cr[0] for cr in cols_rows]

        rows = [cr[1] for cr in cols_rows]

        ax.plot(cols, rows, color='yellow', linewidth=0.8, alpha=0.9)

    except Exception:

        continue


overlay_path = RUN_DIR / 'overlay_s2_rgb_crowns.png'

fig.tight_layout()

fig.savefig(overlay_path)

plt.close(fig)

print('Saved overlay:', overlay_path)

print('Sentinel derived EPSG:', s2_epsg)

Saved overlay: /Users/hbot07/VS Code/Drone-Phenology-Monitoring/output/sat_data/sit_multi_run/overlay_s2_rgb_crowns.png
Sentinel derived EPSG: 32643


In [6]:
# ---- 3) NDVI time series for ALL crowns (saved to disk) ----

from rasterio import features

# Controls
MAX_DATES: int | None = 20  # preview mode for fast iteration; set None for full run
ALL_TOUCHED = True  # critical for small crowns vs 10m pixels
PAD_PIXELS = 40  # expand window around AOI a bit
MIN_PIXELS_PER_OBS = 3  # drop very tiny zonal samples that are often noisy

# Choose an ID column for crowns
CROWN_ID_COL = 'chain_id' if 'chain_id' in crowns.columns else None
if CROWN_ID_COL is None:
    # Fall back to row index if needed
    crowns = crowns.copy()
    crowns['crown_id'] = np.arange(len(crowns), dtype=int)
    CROWN_ID_COL = 'crown_id'

print('Using crown id col:', CROWN_ID_COL)

# Optionally reduce to one item per date (pick lowest cloud_cover for that date)
items_df_use = items_df.copy()
items_df_use = items_df_use.sort_values(['datetime', 'cloud_cover']).dropna(subset=['datetime'])
items_df_use = items_df_use.groupby('datetime', as_index=False).first()
items_df_use = items_df_use.sort_values(['datetime']).reset_index(drop=True)

if MAX_DATES is not None and len(items_df_use) > MAX_DATES:
    idxs = np.linspace(0, len(items_df_use) - 1, MAX_DATES).round().astype(int)
    items_df_use = items_df_use.iloc[idxs].reset_index(drop=True)

id_to_item = {it.id: it for it in items}
items_use = [id_to_item[iid] for iid in items_df_use['id'].tolist() if iid in id_to_item]
print('items_used_for_all_crowns_ts:', len(items_use))
if not items_use:
    raise RuntimeError('No items selected for NDVI extraction after filtering.')

# Reproject crowns into Sentinel CRS once (for this tile/year it should match crowns CRS already)
fallback_lat = float(centroid_wgs84.y)
s2_epsg_common = _sentinel_epsg_from_item(items_use[0], fallback_lat=fallback_lat)
s2_crs_str = f'EPSG:{int(s2_epsg_common)}'
crowns_s2 = crowns.to_crs(s2_crs_str).copy()

# Compute AOI window bounds in Sentinel CRS using pyproj
from pyproj import Transformer
minlon, minlat, maxlon, maxlat = bbox_wgs84
tx = Transformer.from_crs('EPSG:4326', s2_crs_str, always_xy=True)
xs, ys = tx.transform([minlon, minlon, maxlon, maxlon], [minlat, maxlat, minlat, maxlat])
aoi_bounds_s2 = (float(min(xs)), float(min(ys)), float(max(xs)), float(max(ys)))

# Helper: compute NDVI raster for the AOI window
def _read_ndvi_window(item):
    href_red = _signed_href(item, 'B04')
    href_nir = _signed_href(item, 'B08')
    epsg = _sentinel_epsg_from_item(item, fallback_lat=fallback_lat)
    crs_str = f'EPSG:{int(epsg)}'
    if crs_str != s2_crs_str:
        # This should not happen for a single-tile AOI, but keep it explicit.
        raise RuntimeError(f'Unexpected CRS change across items: {crs_str} vs {s2_crs_str}')

    with rasterio.open(href_red) as src_red, rasterio.open(href_nir) as src_nir:
        win = rasterio.windows.from_bounds(*aoi_bounds_s2, transform=src_red.transform)
        win = win.round_offsets().round_lengths()
        win = rasterio.windows.Window(
            col_off=max(0, win.col_off - PAD_PIXELS),
            row_off=max(0, win.row_off - PAD_PIXELS),
            width=min(src_red.width - max(0, win.col_off - PAD_PIXELS), win.width + 2 * PAD_PIXELS),
            height=min(src_red.height - max(0, win.row_off - PAD_PIXELS), win.height + 2 * PAD_PIXELS),
        )
        red = src_red.read(1, window=win, masked=True).astype('float32')
        nir = src_nir.read(1, window=win, masked=True).astype('float32')
        # Combined mask
        mask = np.ma.getmaskarray(red) | np.ma.getmaskarray(nir)
        denom = (nir + red)
        mask |= ~np.isfinite(denom) | (denom == 0)
        ndvi = (nir - red) / denom
        ndvi = np.where(mask, np.nan, np.asarray(ndvi, dtype='float32'))
        transform = rasterio.windows.transform(win, src_red.transform)
        return ndvi, transform, win

# Helper: compute mean NDVI for each crown polygon on an NDVI array
def _zonal_mean_all_crowns(ndvi: np.ndarray, transform, gdf_s2: gpd.GeoDataFrame):
    out = np.full(len(gdf_s2), np.nan, dtype='float32')
    npx = np.zeros(len(gdf_s2), dtype='int32')
    finite = np.isfinite(ndvi)
    for i, geom in enumerate(gdf_s2.geometry):
        if geom is None or geom.is_empty:
            continue
        m = features.geometry_mask(
            [mapping(geom)],
            out_shape=ndvi.shape,
            transform=transform,
            invert=True,
            all_touched=ALL_TOUCHED,
        )
        sel = m & finite
        if not np.any(sel):
            continue
        vals = ndvi[sel]
        out[i] = float(np.nanmean(vals))
        npx[i] = int(np.sum(sel))
    return out, npx

records = []
for k, it in enumerate(items_use):
    dt = _item_dt_str(it)
    if dt is None:
        continue
    ndvi, t, _win = _read_ndvi_window(it)
    means, n_pixels = _zonal_mean_all_crowns(ndvi, t, crowns_s2)
    cloud = it.properties.get('eo:cloud_cover')
    for i in range(len(crowns_s2)):
        records.append({
            'date': dt,
            'item_id': it.id,
            'cloud_cover': cloud,
            'derived_epsg': int(s2_epsg_common),
            'crown_id': int(crowns_s2.iloc[i][CROWN_ID_COL]),
            'ndvi_mean': float(means[i]) if np.isfinite(means[i]) else np.nan,
            'n_pixels': int(n_pixels[i]),
        })
    if (k + 1) % 5 == 0:
        print(f'Processed {k+1}/{len(items_use)} items...')

all_df = pd.DataFrame.from_records(records)
all_df['date'] = pd.to_datetime(all_df['date'])

# Drop highly unstable samples with too few covered pixels.
all_df.loc[pd.to_numeric(all_df['n_pixels'], errors='coerce') < int(MIN_PIXELS_PER_OBS), 'ndvi_mean'] = np.nan

# Bound NDVI into a physically plausible range to reduce visual outliers.
all_df['ndvi_mean'] = pd.to_numeric(all_df['ndvi_mean'], errors='coerce')
all_df.loc[(all_df['ndvi_mean'] < -0.2) | (all_df['ndvi_mean'] > 1.0), 'ndvi_mean'] = np.nan

all_df['year'] = all_df['date'].dt.year.astype(int)
iso = all_df['date'].dt.isocalendar()
all_df['iso_year'] = iso.year.astype(int)
all_df['iso_week'] = iso.week.astype(int)
all_df = all_df.sort_values(['crown_id', 'date']).reset_index(drop=True)

all_csv = RUN_DIR / 'ndvi_timeseries_all_crowns.csv'
all_df.to_csv(all_csv, index=False)
print('Wrote:', all_csv)

# QC: how many valid observations per crown?
valid_counts = (all_df.dropna(subset=['ndvi_mean'])
                .groupby('crown_id')['ndvi_mean']
                .size()
                .rename('n_obs')
                .reset_index()
                .sort_values('n_obs', ascending=False))
counts_csv = RUN_DIR / 'ndvi_valid_counts_by_crown.csv'
valid_counts.to_csv(counts_csv, index=False)
print('Wrote:', counts_csv)

# Per-crown weekly harmonization across years: robust (median + IQR).
weekly_per_crown = (
    all_df.dropna(subset=['ndvi_mean'])
    .groupby(['crown_id', 'iso_week'], as_index=False)
    .agg(
        ndvi_weekly_median=('ndvi_mean', 'median'),
        ndvi_weekly_p25=('ndvi_mean', lambda s: float(np.nanquantile(s, 0.25))),
        ndvi_weekly_p75=('ndvi_mean', lambda s: float(np.nanquantile(s, 0.75))),
        n_obs=('ndvi_mean', 'size'),
        n_years=('year', 'nunique'),
    )
)
weekly_per_crown = weekly_per_crown[weekly_per_crown['n_obs'] >= int(WEEKLY_MIN_OBS_PER_CROWN_WEEK)].copy()
weekly_csv = RUN_DIR / 'ndvi_weekly_harmonized_per_crown.csv'
weekly_per_crown.to_csv(weekly_csv, index=False)
print('Wrote:', weekly_csv)

# Aggregate seasonal curve across all crowns (and years) for cleaner signal diagnostics.
weekly_global = (
    all_df.dropna(subset=['ndvi_mean'])
    .groupby('iso_week', as_index=False)
    .agg(
        ndvi_median=('ndvi_mean', 'median'),
        ndvi_p25=('ndvi_mean', lambda s: float(np.nanquantile(s, 0.25))),
        ndvi_p75=('ndvi_mean', lambda s: float(np.nanquantile(s, 0.75))),
        n_obs=('ndvi_mean', 'size'),
        n_crowns=('crown_id', 'nunique'),
        n_years=('year', 'nunique'),
    )
)
weekly_global_csv = RUN_DIR / 'ndvi_weekly_harmonized_global.csv'
weekly_global.to_csv(weekly_global_csv, index=False)
print('Wrote:', weekly_global_csv)

# Year-wise weekly curve (lets you compare years + harmonized signal quality).
weekly_by_year = (
    all_df.dropna(subset=['ndvi_mean'])
    .groupby(['year', 'iso_week'], as_index=False)
    .agg(
        ndvi_median=('ndvi_mean', 'median'),
        n_obs=('ndvi_mean', 'size'),
        n_crowns=('crown_id', 'nunique'),
    )
)
weekly_by_year_csv = RUN_DIR / 'ndvi_weekly_by_year.csv'
weekly_by_year.to_csv(weekly_by_year_csv, index=False)
print('Wrote:', weekly_by_year_csv)

# QC plot: a handful of crowns with most observations (smoothed).
top_ids = valid_counts.head(12)['crown_id'].tolist()
fig, ax = plt.subplots(figsize=(10, 4), dpi=160)
for cid in top_ids:
    g = all_df[all_df['crown_id'] == cid].dropna(subset=['ndvi_mean']).sort_values('date')
    if g.empty:
        continue
    y = g['ndvi_mean'].rolling(window=3, min_periods=1, center=True).median()
    ax.plot(g['date'], y, linewidth=1.1, alpha=0.85, label=str(cid))
ax.set_title(f'Sentinel-2 NDVI (smoothed) | top crowns by valid obs | {DATASET_NAME} | {"multi-year" if USE_MULTI_YEAR else YEAR}')
ax.set_ylabel('NDVI')
ax.set_xlabel('date')
ax.grid(True, alpha=0.3)
ax.legend(loc='best', fontsize=7, ncol=3, title='crown_id')
fig.tight_layout()
qc_plot = RUN_DIR / 'ndvi_timeseries_top12_crowns_smoothed.png'
fig.savefig(qc_plot)
plt.close(fig)
print('Saved plot:', qc_plot)

# Plot: harmonized weekly NDVI with IQR band across all crowns/years.
fig, ax = plt.subplots(figsize=(10, 4), dpi=160)
wg = weekly_global.sort_values('iso_week')
ax.plot(wg['iso_week'], wg['ndvi_median'], linewidth=1.5, color='tab:green', label='weekly median')
ax.fill_between(wg['iso_week'], wg['ndvi_p25'], wg['ndvi_p75'], color='tab:green', alpha=0.2, label='IQR (p25-p75)')
ax.set_title(f'Harmonized weekly NDVI across crowns | {DATASET_NAME} | {"multi-year" if USE_MULTI_YEAR else YEAR}')
ax.set_ylabel('NDVI')
ax.set_xlabel('ISO week')
ax.set_xlim(1, 53)
ax.grid(True, alpha=0.3)
ax.legend(loc='best', fontsize=8)
fig.tight_layout()
harm_plot = RUN_DIR / 'ndvi_weekly_harmonized_global.png'
fig.savefig(harm_plot)
plt.close(fig)
print('Saved plot:', harm_plot)

# Plot: weekly NDVI by year.
fig, ax = plt.subplots(figsize=(10, 4), dpi=160)
for yy in sorted(weekly_by_year['year'].unique().tolist()):
    gy = weekly_by_year[weekly_by_year['year'] == yy].sort_values('iso_week')
    ax.plot(gy['iso_week'], gy['ndvi_median'], marker='o', markersize=2.2, linewidth=1.0, alpha=0.9, label=str(yy))
ax.set_title(f'Weekly NDVI by year across crowns | {DATASET_NAME}')
ax.set_ylabel('NDVI median')
ax.set_xlabel('ISO week')
ax.set_xlim(1, 53)
ax.grid(True, alpha=0.3)
ax.legend(loc='best', fontsize=8, ncol=2, title='year')
fig.tight_layout()
by_year_plot = RUN_DIR / 'ndvi_weekly_by_year.png'
fig.savefig(by_year_plot)
plt.close(fig)
print('Saved plot:', by_year_plot)

Using crown id col: chain_id
items_used_for_all_crowns_ts: 20
Processed 5/20 items...
Processed 10/20 items...
Processed 15/20 items...
Processed 20/20 items...
Wrote: /Users/hbot07/VS Code/Drone-Phenology-Monitoring/output/sat_data/sit_multi_run/ndvi_timeseries_all_crowns.csv
Wrote: /Users/hbot07/VS Code/Drone-Phenology-Monitoring/output/sat_data/sit_multi_run/ndvi_valid_counts_by_crown.csv
Wrote: /Users/hbot07/VS Code/Drone-Phenology-Monitoring/output/sat_data/sit_multi_run/ndvi_weekly_harmonized_per_crown.csv
Wrote: /Users/hbot07/VS Code/Drone-Phenology-Monitoring/output/sat_data/sit_multi_run/ndvi_weekly_harmonized_global.csv
Wrote: /Users/hbot07/VS Code/Drone-Phenology-Monitoring/output/sat_data/sit_multi_run/ndvi_weekly_by_year.csv
Saved plot: /Users/hbot07/VS Code/Drone-Phenology-Monitoring/output/sat_data/sit_multi_run/ndvi_timeseries_top12_crowns_smoothed.png
Saved plot: /Users/hbot07/VS Code/Drone-Phenology-Monitoring/output/sat_data/sit_multi_run/ndvi_weekly_harmonized_globa

In [7]:
# ---- 4) Example-crown NDVI signals across years (10 crowns, non-median-across-crowns) ----

# This plot shows per-crown temporal signals, split by year.
# Within each crown/year we summarize multiple observations in the same ISO week
# using a weekly median (for that crown only) to reduce high-frequency noise.

N_EXAMPLE_CROWNS = 10

if 'all_df' not in globals():
    raise RuntimeError('all_df not found. Run Cell 5 first.')

df_plot = all_df.copy()
df_plot['ndvi_mean'] = pd.to_numeric(df_plot['ndvi_mean'], errors='coerce')
df_plot = df_plot.dropna(subset=['ndvi_mean']).copy()

if df_plot.empty:
    raise RuntimeError('No NDVI observations available to plot.')

# Pick crowns with the best temporal coverage.
crown_rank = (
    df_plot.groupby('crown_id', as_index=False)
    .agg(n_obs=('ndvi_mean', 'size'), n_years=('year', 'nunique'))
    .sort_values(['n_years', 'n_obs'], ascending=[False, False])
)
example_crowns = crown_rank.head(int(N_EXAMPLE_CROWNS))['crown_id'].tolist()

df_ex = df_plot[df_plot['crown_id'].isin(example_crowns)].copy()
weekly = (
    df_ex.groupby(['crown_id', 'year', 'iso_week'], as_index=False)
    .agg(ndvi_week=('ndvi_mean', 'median'), n_obs=('ndvi_mean', 'size'))
)

if weekly.empty:
    raise RuntimeError('Weekly crown-year NDVI table is empty.')

# 2x5 panel for 10 crowns
ncols = 5
nrows = int(np.ceil(len(example_crowns) / ncols))
fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(20, 3.8 * nrows), dpi=160, sharex=True, sharey=True)
axes = np.array(axes).reshape(-1)

years_sorted = sorted(weekly['year'].unique().tolist())
cmap = plt.cm.get_cmap('tab10', max(3, len(years_sorted)))
year_to_color = {yy: cmap(i % cmap.N) for i, yy in enumerate(years_sorted)}

for i, cid in enumerate(example_crowns):
    ax = axes[i]
    g = weekly[weekly['crown_id'] == cid].sort_values(['year', 'iso_week'])
    for yy in years_sorted:
        gy = g[g['year'] == yy]
        if gy.empty:
            continue
        ax.plot(
            gy['iso_week'],
            gy['ndvi_week'],
            marker='o',
            markersize=2.3,
            linewidth=1.0,
            alpha=0.9,
            color=year_to_color[yy],
            label=str(yy),
        )
    obs = int(crown_rank.loc[crown_rank['crown_id'] == cid, 'n_obs'].iloc[0])
    nyr = int(crown_rank.loc[crown_rank['crown_id'] == cid, 'n_years'].iloc[0])
    ax.set_title(f'crown {cid} | obs={obs} | years={nyr}', fontsize=9)
    ax.set_xlim(1, 53)
    ax.grid(True, alpha=0.25)

# Hide unused axes if < 10 crowns are available
for j in range(len(example_crowns), len(axes)):
    axes[j].axis('off')

for ax in axes[:len(example_crowns)]:
    ax.set_xlabel('ISO week')
    ax.set_ylabel('NDVI')

handles = [plt.Line2D([0], [0], color=year_to_color[yy], lw=2) for yy in years_sorted]
labels = [str(yy) for yy in years_sorted]
fig.legend(handles, labels, title='year', loc='upper center', ncol=min(6, len(labels)), frameon=False)

fig.suptitle(f'NDVI signals across years for {len(example_crowns)} example crowns | {DATASET_NAME}', y=0.995, fontsize=12)
fig.tight_layout(rect=[0, 0, 1, 0.96])

example_plot = RUN_DIR / 'ndvi_example_10_crowns_across_years.png'
fig.savefig(example_plot)
plt.close(fig)
print('Saved plot:', example_plot)

# Export the plotted data for inspection.
example_csv = RUN_DIR / 'ndvi_example_10_crowns_across_years.csv'
weekly.to_csv(example_csv, index=False)
print('Wrote:', example_csv)

/var/folders/cx/8v11zwh97cxgdgt_ph13_56r0000gn/T/ipykernel_17710/849906749.py:43: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = plt.cm.get_cmap('tab10', max(3, len(years_sorted)))


Saved plot: /Users/hbot07/VS Code/Drone-Phenology-Monitoring/output/sat_data/sit_multi_run/ndvi_example_10_crowns_across_years.png
Wrote: /Users/hbot07/VS Code/Drone-Phenology-Monitoring/output/sat_data/sit_multi_run/ndvi_example_10_crowns_across_years.csv
